<a href="https://colab.research.google.com/github/juanpajaro/aprendizaje_profundo_salud_puj_2026/blob/main/hugging_apnea_frozen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate

In [ ]:
!pip install --upgrade transformers

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from evaluate import load
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
#Se carga Drive al ambiente en colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#si existe GPU se activaran, si no quedará en cpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
#Directorio de los datos
TRAIN_PATH = "/content/drive/MyDrive/datos_apnea/datos_apnea_small_train.csv"
VAL_PATH = "/content/drive/MyDrive/datos_apnea/datos_apnea_small_val.csv"
TEST_PATH = "/content/drive/MyDrive/datos_apnea/datos_apnea_small_test.csv"

In [ ]:
#Se cargan en pandas
df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)

In [ ]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

In [ ]:
def preprocess_df(df):
    """Ensures types are correct and removes rows with empty text."""
    df['EnfermedadActual'] = df['EnfermedadActual'].astype(str)
    df['Apnea'] = df['Apnea'].astype(int)
    return df

In [ ]:
df_train = preprocess_df(df_train)
df_val = preprocess_df(df_val)
df_test = preprocess_df(df_test)

In [ ]:
#se convierte el dataset de pandas a la estructura de huggingface
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(df_train.reset_index(drop=True)),
    "validation": Dataset.from_pandas(df_val.reset_index(drop=True)),
    "test": Dataset.from_pandas(df_test.reset_index(drop=True))
})

In [ ]:
#se cargar el tokenizador, es decir el preprocesamiento dle texto. Para convertir el texto a secuencia de numeros, en conjunto con los tokens especiales.
model_ckpt = "PlanTL-GOB-ES/roberta-base-biomedical-es"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
def tokenize_function(examples):
    # Truncate clinical narratives to RoBERTa's maximum limit of 512 tokens
    return tokenizer(
        examples["EnfermedadActual"],
        truncation=True,
        max_length=256 #maximo valor 512
    )

In [ ]:
#se usa el tokenizador. Función map para que no se sobre cargue la memoría.
#tiempo estimado 10 minutos.
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

In [ ]:
print(tokenized_datasets["train"].shape)
print(tokenized_datasets["validation"].shape)
print(tokenized_datasets["test"].shape)

In [ ]:
print(tokenized_datasets["train"][0])

In [ ]:
print(tokenized_datasets["validation"][0])

In [ ]:
#renombrar las columnas para cumplir con la estructura de hugging face
for split in tokenized_datasets.keys():
    tokenized_datasets[split] = tokenized_datasets[split].rename_column("Apnea", "label")
    tokenized_datasets[split].set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2).to(device)

In [ ]:
#Se cargan los parametros congelados
for param in model.parameters():
    param.requires_grad = False

In [ ]:
#Se libera la cabeza final (la que mapea los embedding a la clasificacion)
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
#Se liberan las últimas capas. Roberta al igual que la mayoría de los modelos BERT tienen doce capas.
layers_to_unfreeze = ["layer.9", "layer.10", "layer.11"]

In [ ]:
for name, param in model.roberta.encoder.layer.named_parameters():
    if any(layer_id in name for layer_id in layers_to_unfreeze):
        param.requires_grad = True

In [ ]:
#Verificacion de los parametros
print("--- Parameter Status Verification ---")
total_params = 0
trainable_params = 0
for name, param in model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"TRAINABLE: {name}")

print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
#Padding dinamico
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Se definen metricas de evaluacion
metric_acc = load("accuracy")
metric_f1 = load("f1")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {"accuracy": acc, "f1": f1}

In [ ]:
#Donde guardar los datos
#importante guardar en una carpeta de drive si despues se quiere usar
output_dir = "/content/"

In [ ]:
#Se cargan los hiperparametros del entrenamiento
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=8,       # Adjust down to 8 or 4 if you encounter an OOM error
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",          # Prioritizes the macro F1-score for medical class imbalances
    logging_steps=10,
    fp16=torch.cuda.is_available(),       # Accelerates training on Colab's T4/A100 GPUs
    report_to="none"
)

In [ ]:
#Se carga la configuracion del modelo.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
print("Starting training...")
trainer.train()

In [ ]:
#evaluacion en test
print("\nEvaluating best model performance on the Test Split...")
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test Macro F1-Score: {test_results['eval_f1']:.4f}")